In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
BRONZE_PATH = "workspace.case_spark_cvm.bronze_registro_subclasse_cvm"
NOME_TABELA  = f"silver_registro_subclasse_cvm" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## CVM - Fundos Investimentos - Registros Subclasse

In [0]:
df_registro_subclasse_cvm = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento" )

### 1.1 tratemento silver

#### 1.1.1 Retirando dados duplicados

In [0]:
# 1. Definimos a chave correta da tabela Dimensão de Subclasses
chave_negocio = ["ID_Subclasse"]

# 2. Criamos o peso do status (B para ativos, A para inativos)
df_registro_subclasse_cvm = df_registro_subclasse_cvm.withColumn(
    "_peso_status",
    f.when(f.col("Situacao").isin("Em Funcionamento Normal", "Fase Pré-Operacional"), f.lit("B"))
     .otherwise(f.lit("A"))
)

# 3. Criamos a super-chave usando a data de início da situação
df_registro_subclasse_cvm = df_registro_subclasse_cvm.withColumn(
    "_ordem_desempate",
    f.concat_ws("_", f.col("Data_Inicio_Situacao"), f.col("_peso_status"))
)

# 4. Espalhamos a data vencedora para o filtro da quarentena
window_subclasse = Window.partitionBy(chave_negocio)

df_registro_subclasse_cvm = df_registro_subclasse_cvm.withColumn(
    "_max_ordem_desempate", 
    f.max("_ordem_desempate").over(window_subclasse)
).withColumn(
    "_data_oficial",
    f.max(
        f.when(f.col("_ordem_desempate") == f.col("_max_ordem_desempate"), f.col("Data_Inicio_Situacao"))
    ).over(window_subclasse)
)

# 5. Aplicamos a SUA função com a super-chave
df_registro_subclasse_cvm, df_todas_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_registro_subclasse_cvm,
    chave_negocio=chave_negocio,
    coluna_ordenacao="_ordem_desempate" # <--- AQUI VAI A COLUNA QUE CRIAMOS!
)

# 6. Filtro Anti-Spam (só aciona se a CVM mandar 2 registros iguais no mesmo dia)
df_quarentena_real = (df_todas_duplicadas
    .filter(f.col("Data_Inicio_Situacao") == f.col("_data_oficial"))
    .withColumn("_motivo_quarentena", f.lit("Anomalia CVM: Múltiplos registros conflitantes para a mesma Subclasse na mesma data"))
)

# Limpeza e Gravação
colunas_sujeira = ["_peso_status", "_ordem_desempate", "_data_oficial"]
df_registro_subclasse_cvm = df_registro_subclasse_cvm.drop(*colunas_sujeira)
df_quarentena_real = df_quarentena_real.drop(*colunas_sujeira)

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_real, 
    tabela_origem="bronze_registro_subclasse_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
regras_qualidade = {
    "ID_Registro_Classe": "not_null",    # Não pode ser vazio (Substitui o dropna)
    "ID_Subclasse": "not_null",          # Não pode ser vazio (Substitui o dropna)
}

df_registro_subclasse_cvm, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_registro_subclasse_cvm,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_registro_subclasse_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_registro_subclasse_cvm = df_registro_subclasse_cvm.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm.select(
    # 1. Chaves de Identificação e Relacionamento (Hierarquia CVM 175)
    f.col('ID_Registro_Classe').cast(t.IntegerType()).alias('id_registro_classe'), # Chave Estrangeira (FK)
    f.col('ID_Subclasse').cast(t.StringType()).alias('id_subclasse'),             # Chave Primária (PK)
    f.col('Codigo_CVM').cast(t.IntegerType()).alias('codigo_cvm'),
    
    # 2. Ciclo de Vida e Status
    f.col('Data_Constituicao').cast(t.DateType()).alias('data_constituicao'),
    f.col('Data_Inicio').cast(t.DateType()).alias('data_inicio'),
    f.col('Situacao').cast(t.StringType()).alias('situacao'),
    f.col('Data_Inicio_Situacao').cast(t.DateType()).alias('data_inicio_situacao'),
    
    # 3. Características e Estrutura da Emissão (Subclasse)
    f.col('Denominacao_Social').cast(t.StringType()).alias('denominacao_social'),
    f.col('Forma_Condominio').cast(t.StringType()).alias('forma_condominio'),
    f.col('Exclusivo').cast(t.StringType()).alias('exclusivo'),
    f.col('Publico_Alvo').cast(t.StringType()).alias('publico_alvo'),
    f.col('Previdenciario').cast(t.StringType()).alias('previdenciario'),
    f.col('Exclusivo_INR').cast(t.StringType()).alias('exclusivo_inr'),
    f.col('Exclusivo_Previdencia_Complementar').cast(t.StringType()).alias('exclusivo_previdencia_complementar')
)

### 1.2 Salvar na camada Silver


In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["id_subclasse"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_registro_subclasse_cvm, 
    tabela_destino= SILVER_PATH, 
    chave_negocio=chave_negocio
    )